# ⬡ Portfolio Optimization Intelligence System
## Week 5: Modern Portfolio Theory (Markowitz)

**Team:** Ashwini | Dibyendu Sarkar | Jyoti Ranjan Sethi  
**Institution:** IIT Madras · Multi-Agent Systems · 2026

---

### 📁 File Structure
```
portfolio_optimizer/
├── Portfolio_Optimization_Agent_Week5.ipynb  ← YOU ARE HERE
├── requirements.txt
├── app.py                          (Week 3 dashboard)
├── app_trial.py                    (Week 4 dashboard)
├── agents/
│   ├── __init__.py
│   ├── market_data/
│   │   ├── __init__.py
│   │   ├── agent.py
│   │   ├── storage.py
│   │   ├── validator.py
│   │   └── api.py
│   ├── risk_management/
│   │   ├── __init__.py
│   │   ├── agent.py
│   │   └── api.py
│   └── alpha_signal/
│       ├── indicators.py
│       ├── signal_generator.py
│       ├── backtester.py
│       ├── agent.py           (Portfolio Optimization Agent)
│       └── api.py             (Portfolio Optimization API)
└── agent_ui/
    ├── risk_ui.py
    ├── alpha_ui.py
    └── streamlit_app.py
```

---

### 📋 This Notebook Covers
1. **Setup & Installation**
2. **Market Data Agent** — fetching & storing price data
3. **Risk Management Agent** — VaR, CVaR, Sharpe, Drawdown
4. **Technical Indicators** — SMA, EMA, RSI, MACD, Bollinger Bands
5. **Signal Generator** — Buy/Sell/Hold recommendations
6. **Portfolio Optimization Agent** — Markowitz mean-variance optimization
7. **Efficient Frontier** — visualization of optimal portfolios
8. **REST API** — FastAPI endpoints
9. **Gradio Dashboard** — interactive UI
10. **Full Integration Demo**

---
## 📦 Section 1: Setup & Installation

In [ ]:
# Install all required packages
!pip install yfinance pandas numpy scipy fastapi uvicorn pydantic requests \
             plotly gradio python-dateutil pytz streamlit --quiet

print("✅ All packages installed successfully!")

In [ ]:
# Core imports used throughout this notebook
import numpy as np
import pandas as pd
import requests
import warnings
import os
import sys
import time
import threading
import sqlite3
from datetime import datetime, timedelta
from typing import Dict, List, Tuple, Optional

import yfinance as yf
from scipy.optimize import minimize
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

print("✅ All imports successful!")
print(f"   NumPy:  {np.__version__}")
print(f"   Pandas: {pd.__version__}")
print(f"   yfinance available: {yf.__version__ if hasattr(yf, '__version__') else 'yes'}")

---
## 🏪 Section 2: Market Data Agent
### `agents/market_data/agent.py`

Fetches real-time and historical price data from Yahoo Finance.

In [ ]:
# ============================================================
# agents/market_data/agent.py
# ============================================================

class MarketDataAgent:
    """
    Market Data Agent — Week 2
    Fetches and manages market data from Yahoo Finance.
    Foundation agent for the multi-agent system.
    """

    def __init__(self, symbols: List[str]):
        self.symbols    = symbols
        self.data_cache = {}
        self.last_update = {}

    def fetch_realtime_data(self, symbol: str) -> Dict:
        """Fetch current market data for a symbol via yfinance."""
        try:
            ticker = yf.Ticker(symbol)
            info   = ticker.info
            data = {
                'symbol':         symbol,
                'timestamp':      datetime.now().isoformat(),
                'price':          info.get('currentPrice', info.get('regularMarketPrice')),
                'open':           info.get('regularMarketOpen'),
                'high':           info.get('dayHigh'),
                'low':            info.get('dayLow'),
                'volume':         info.get('volume'),
                'market_cap':     info.get('marketCap'),
                'pe_ratio':       info.get('trailingPE'),
                'dividend_yield': info.get('dividendYield'),
            }
            print(f"  ✓ {symbol}: ${data['price']}")
            return data
        except Exception as e:
            print(f"  ✗ Error fetching {symbol}: {e}")
            return None

    def fetch_historical_data(self, symbol: str, period: str = '1y') -> pd.DataFrame:
        """
        Fetch OHLCV historical data.
        period: 1d | 5d | 1mo | 3mo | 6mo | 1y | 2y | 5y | max
        """
        try:
            ticker = yf.Ticker(symbol)
            df     = ticker.history(period=period)
            print(f"  ✓ {symbol}: {len(df)} days of historical data")
            return df
        except Exception as e:
            print(f"  ✗ Error fetching historical data for {symbol}: {e}")
            return None

    def fetch_all_symbols(self) -> Dict[str, Dict]:
        """Fetch real-time data for all tracked symbols."""
        results = {}
        for symbol in self.symbols:
            data = self.fetch_realtime_data(symbol)
            if data:
                results[symbol] = data
                self.data_cache[symbol]  = data
                self.last_update[symbol] = datetime.now()
            time.sleep(0.3)   # polite rate limiting
        return results

    def get_latest_price(self, symbol: str) -> Optional[float]:
        """Return latest cached price (None if not yet fetched)."""
        return self.data_cache.get(symbol, {}).get('price')


# ── Quick demo ──────────────────────────────────────────────
SYMBOLS = ['AAPL', 'GOOGL', 'MSFT', 'TSLA', 'AMZN']
market_agent = MarketDataAgent(SYMBOLS)

print("Fetching real-time prices...")
realtime_data = market_agent.fetch_all_symbols()
print("\n✅ MarketDataAgent ready!")

In [ ]:
# ============================================================
# agents/market_data/storage.py  — Thread-safe SQLite storage
# ============================================================

class MarketDataStorage:
    """Thread-safe SQLite storage for market data."""

    def __init__(self, db_path: str = 'market_data.db'):
        self.db_path = db_path
        self.local   = threading.local()
        self._create_tables()

    def _get_conn(self):
        if not hasattr(self.local, 'conn'):
            self.local.conn = sqlite3.connect(self.db_path, check_same_thread=False)
        return self.local.conn

    def _create_tables(self):
        conn   = self._get_conn()
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS realtime_prices (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                symbol TEXT NOT NULL,
                timestamp TEXT NOT NULL,
                price REAL NOT NULL,
                open REAL, high REAL, low REAL,
                volume INTEGER, market_cap REAL, pe_ratio REAL
            )"""
        )
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS historical_prices (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                symbol TEXT NOT NULL,
                date TEXT NOT NULL,
                open REAL, high REAL, low REAL, close REAL NOT NULL, volume INTEGER,
                UNIQUE(symbol, date)
            )"""
        )
        conn.commit()

    def save_realtime_data(self, data: Dict):
        conn   = self._get_conn()
        cursor = conn.cursor()
        try:
            cursor.execute("""
                INSERT INTO realtime_prices
                (symbol,timestamp,price,open,high,low,volume,market_cap,pe_ratio)
                VALUES (?,?,?,?,?,?,?,?,?)""",
                (data['symbol'], data['timestamp'], data['price'],
                 data.get('open'), data.get('high'), data.get('low'),
                 data.get('volume'), data.get('market_cap'), data.get('pe_ratio'))
            )
            conn.commit()
        except Exception as e:
            print(f"  Storage error: {e}")
            conn.rollback()

    def save_historical_data(self, symbol: str, df: pd.DataFrame):
        conn   = self._get_conn()
        cursor = conn.cursor()
        try:
            for date, row in df.iterrows():
                cursor.execute("""
                    INSERT OR REPLACE INTO historical_prices
                    (symbol,date,open,high,low,close,volume)
                    VALUES (?,?,?,?,?,?,?)""",
                    (symbol, date.strftime('%Y-%m-%d'),
                     row['Open'], row['High'], row['Low'], row['Close'], row['Volume'])
                )
            conn.commit()
        except Exception as e:
            print(f"  Storage error: {e}")
            conn.rollback()

    def get_latest_prices(self, symbols: List[str] = None) -> pd.DataFrame:
        conn  = self._get_conn()
        query = """
            SELECT symbol, price, timestamp FROM realtime_prices
            WHERE id IN (SELECT MAX(id) FROM realtime_prices GROUP BY symbol)"""
        return pd.read_sql_query(query, conn)


storage = MarketDataStorage()

# Persist realtime data
for sym, d in realtime_data.items():
    storage.save_realtime_data(d)

print("\nLatest prices from DB:")
print(storage.get_latest_prices())

In [ ]:
# ============================================================
# agents/market_data/validator.py
# ============================================================

class DataValidator:
    """Validates and cleans market data before storage."""

    @staticmethod
    def validate_realtime_data(data: Dict) -> Tuple[bool, List[str]]:
        errors = []
        for field in ['symbol', 'timestamp', 'price', 'volume']:
            if field not in data or data[field] is None:
                errors.append(f"Missing/null: {field}")
        if 'price' in data and data['price'] is not None and data['price'] <= 0:
            errors.append(f"Invalid price: {data['price']}")
        if 'volume' in data and data['volume'] is not None and data['volume'] < 0:
            errors.append(f"Invalid volume: {data['volume']}")
        return len(errors) == 0, errors

    @staticmethod
    def clean_historical_data(df: pd.DataFrame) -> pd.DataFrame:
        df = df.dropna(subset=['Close'])
        df['price_change'] = df['Close'].pct_change()
        df = df[abs(df['price_change']) < 0.5]   # remove >50% daily moves
        df = df.ffill()
        return df

    @staticmethod
    def calculate_data_quality_score(df: pd.DataFrame) -> float:
        score = 100.0
        missing_pct  = df.isnull().sum().sum() / (len(df) * len(df.columns))
        score       -= missing_pct * 30
        dup_pct      = df.index.duplicated().sum() / len(df)
        score       -= dup_pct * 20
        if 'Close' in df.columns:
            extreme = (abs(df['Close'].pct_change().dropna()) > 0.2).sum()
            score  -= (extreme / len(df)) * 20
        return max(0, score)


validator = DataValidator()

# Validate first fetched record
first_sym  = list(realtime_data.keys())[0]
is_valid, errors = validator.validate_realtime_data(realtime_data[first_sym])
print(f"Validation for {first_sym}: valid={is_valid}, errors={errors}")

---
## 🛡️ Section 3: Risk Management Agent
### `agents/risk_management/agent.py`

In [ ]:
# ============================================================
# agents/risk_management/agent.py
# ============================================================

class RiskManagementAgent:
    """
    Risk Management Agent (RiskIQ) — Week 3
    Calculates VaR, CVaR, Volatility, Max Drawdown,
    Sharpe Ratio, Beta, Stress Tests, Correlation Matrix.
    """

    def __init__(self, market_data_api: str = 'http://localhost:8000'):
        self.market_data_api = market_data_api
        self.risk_limits = {
            'var_95':      0.05,
            'var_99':      0.10,
            'volatility':  0.30,
            'max_drawdown':0.20,
            'min_sharpe':  1.0,
        }

    # ── Internal helper: get returns from yfinance directly ──
    def _get_returns_direct(self, symbol: str, period: str = '1y') -> pd.Series:
        """Fetch returns directly from yfinance (fallback when API not running)."""
        try:
            df = yf.Ticker(symbol).history(period=period)
            return df['Close'].pct_change().dropna()
        except:
            return None

    def fetch_historical_returns(self, symbol: str, period: str = '1y') -> pd.Series:
        """Fetch returns — tries REST API first, falls back to direct yfinance."""
        try:
            r = requests.get(
                f"{self.market_data_api}/historical/{symbol}",
                params={'period': period}, timeout=5
            )
            if r.status_code == 200:
                df = pd.DataFrame(r.json())
                if 'Date' in df.columns:
                    df['Date'] = pd.to_datetime(df['Date'])
                    df = df.set_index('Date')
                return df['Close'].pct_change().dropna()
        except:
            pass
        return self._get_returns_direct(symbol, period)

    # ── VaR ──────────────────────────────────────────────────
    def calculate_var(self, returns: pd.Series, confidence: float = 0.95,
                      portfolio_value: float = 100_000) -> Dict:
        """
        Value at Risk (Historical Simulation)
        Formula: VaR = percentile(losses, 1-confidence)
        """
        if returns is None or len(returns) == 0:
            return None
        var_pct    = abs(np.percentile(returns, (1 - confidence) * 100))
        var_dollar = var_pct * portfolio_value
        return {
            'confidence':      confidence,
            'var_pct':         var_pct,
            'var_dollar':      var_dollar,
            'interpretation':  (
                f"We are {confidence*100:.0f}% confident we won't lose more than "
                f"${var_dollar:,.0f} ({var_pct:.2%}) in one day"
            )
        }

    # ── CVaR ─────────────────────────────────────────────────
    def calculate_cvar(self, returns: pd.Series, confidence: float = 0.95,
                       portfolio_value: float = 100_000) -> Dict:
        """
        Conditional VaR / Expected Shortfall
        Formula: CVaR = E[Loss | Loss > VaR]
        """
        if returns is None or len(returns) == 0:
            return None
        var_r       = -self.calculate_var(returns, confidence, portfolio_value)['var_pct']
        tail_losses = returns[returns <= var_r]
        cvar_pct    = abs(tail_losses.mean())
        return {
            'confidence':     confidence,
            'cvar_pct':       cvar_pct,
            'cvar_dollar':    cvar_pct * portfolio_value,
            'num_tail_events':len(tail_losses),
            'interpretation': (
                f"If losses exceed VaR, average loss is "
                f"${cvar_pct*portfolio_value:,.0f} ({cvar_pct:.2%})"
            )
        }

    # ── Volatility ───────────────────────────────────────────
    def calculate_volatility(self, returns: pd.Series) -> Dict:
        """
        Annualised Volatility
        Formula: σ_annual = σ_daily × √252
        """
        if returns is None or len(returns) == 0:
            return None
        daily_vol  = returns.std()
        annual_vol = daily_vol * np.sqrt(252)
        return {
            'daily_volatility':  daily_vol,
            'annual_volatility': annual_vol,
            'interpretation':    f"Annual volatility: {annual_vol:.2%}"
        }

    # ── Max Drawdown ─────────────────────────────────────────
    def calculate_max_drawdown(self, returns: pd.Series) -> Dict:
        """
        Maximum Drawdown
        Formula: DD_t = (P_t - max(P_0..P_t)) / max(P_0..P_t)
        """
        if returns is None or len(returns) == 0:
            return None
        cum_ret     = (1 + returns).cumprod()
        running_max = cum_ret.cummax()
        drawdown    = (cum_ret - running_max) / running_max
        max_dd      = abs(drawdown.min())
        return {
            'max_drawdown':     max_dd,
            'max_drawdown_pct': max_dd,
            'interpretation':   f"Worst peak-to-trough decline: {max_dd:.2%}"
        }

    # ── Sharpe Ratio ─────────────────────────────────────────
    def calculate_sharpe_ratio(self, returns: pd.Series,
                                risk_free_rate: float = 0.04) -> Dict:
        """
        Sharpe Ratio
        Formula: S = (R_p - R_f) / σ_p
        """
        if returns is None or len(returns) == 0:
            return None
        n_years      = len(returns) / 252
        total_ret    = (1 + returns).prod() - 1
        ann_ret      = (1 + total_ret) ** (1 / max(n_years, 0.01)) - 1
        ann_vol      = returns.std() * np.sqrt(252)
        sharpe       = (ann_ret - risk_free_rate) / ann_vol if ann_vol else 0
        rating = ("Exceptional" if sharpe > 3 else "Very Good" if sharpe > 2
                  else "Good" if sharpe > 1 else "Acceptable" if sharpe > 0.5
                  else "Poor" if sharpe > 0 else "Losing Money")
        return {
            'sharpe_ratio':         sharpe,
            'annualized_return':    ann_ret,
            'annualized_volatility':ann_vol,
            'risk_free_rate':       risk_free_rate,
            'rating':               rating,
        }

    # ── Stress Test ──────────────────────────────────────────
    def stress_test(self, returns: pd.Series, portfolio_value: float = 100_000,
                    scenarios: Dict = None) -> Dict:
        """Stress test portfolio under historical crash scenarios."""
        if scenarios is None:
            scenarios = {
                'Moderate Decline -5%':  -0.05,
                'Correction -10%':       -0.10,
                'Bear Market -20%':      -0.20,
                'Severe Crash -30%':     -0.30,
                '2008 Crisis -50%':      -0.50,
                'COVID Crash -35%':      -0.35,
                'Flash Crash -10%':      -0.10,
                'Rate Shock -15%':       -0.15,
            }
        avg_daily = returns.mean() if returns is not None and len(returns) > 0 else 0.0003
        results   = {}
        for name, shock in scenarios.items():
            shocked_val = portfolio_value * (1 + shock)
            loss        = portfolio_value - shocked_val
            days_rec    = abs(shock) / avg_daily if avg_daily > 0 else float('inf')
            yrs_rec     = round(days_rec / 252, 1) if days_rec != float('inf') else None
            results[name] = {
                'shock_pct':       shock,
                'shocked_value':   shocked_val,
                'loss_amount':     loss,
                'loss_pct':        shock,
                'years_to_recover':yrs_rec,
            }
        return results

    # ── Correlation Matrix ───────────────────────────────────
    def calculate_correlation_matrix(self, symbols: List[str],
                                     period: str = '1y') -> pd.DataFrame:
        """Compute pairwise correlation matrix for a list of symbols."""
        returns_dict = {}
        for sym in symbols:
            r = self.fetch_historical_returns(sym, period)
            if r is not None:
                returns_dict[sym] = r
        if len(returns_dict) < 2:
            return None
        return pd.DataFrame(returns_dict).dropna().corr()

    # ── Full Assessment ──────────────────────────────────────
    def assess_risk(self, symbol: str, portfolio_value: float = 100_000) -> Dict:
        """Run a complete risk assessment for a single asset."""
        returns = self.fetch_historical_returns(symbol)
        if returns is None:
            return {'error': 'Could not fetch data'}
        return {
            'symbol':         symbol,
            'portfolio_value':portfolio_value,
            'data_points':    len(returns),
            'var_95':         self.calculate_var(returns, 0.95, portfolio_value),
            'var_99':         self.calculate_var(returns, 0.99, portfolio_value),
            'cvar_95':        self.calculate_cvar(returns, 0.95, portfolio_value),
            'volatility':     self.calculate_volatility(returns),
            'max_drawdown':   self.calculate_max_drawdown(returns),
            'sharpe_ratio':   self.calculate_sharpe_ratio(returns),
            'stress_test':    self.stress_test(returns, portfolio_value),
        }


risk_agent = RiskManagementAgent()

print("Running risk assessment for AAPL ...")
assessment = risk_agent.assess_risk('AAPL', portfolio_value=100_000)

if 'error' not in assessment:
    v95 = assessment['var_95']
    vol = assessment['volatility']
    sha = assessment['sharpe_ratio']
    mdd = assessment['max_drawdown']
    print(f"\n{'='*55}")
    print(f"  AAPL Risk Assessment  |  ${assessment['portfolio_value']:,} portfolio")
    print(f"{'='*55}")
    print(f"  Data Points:      {assessment['data_points']} trading days")
    print(f"  VaR 95%:          {v95['var_pct']:.2%}  (${v95['var_dollar']:,.0f}/day)")
    print(f"  Annual Volatility:{vol['annual_volatility']:.2%}")
    print(f"  Sharpe Ratio:     {sha['sharpe_ratio']:.2f}  [{sha['rating']}]")
    print(f"  Max Drawdown:     {mdd['max_drawdown']:.2%}")
    print(f"{'='*55}")

---
## 📊 Section 4: Technical Indicators
### `agents/alpha_signal/indicators.py`

In [ ]:
# ============================================================
# agents/alpha_signal/indicators.py
# ============================================================

class TechnicalIndicators:
    """
    Technical Indicators Calculator — Week 4
    Implements: SMA, EMA, RSI, MACD, Bollinger Bands, Stochastic, ATR
    All methods are static for easy reuse.
    """

    @staticmethod
    def sma(prices: pd.Series, period: int) -> pd.Series:
        """
        Simple Moving Average
        Formula: SMA_n(t) = (P_t + P_{t-1} + ... + P_{t-n+1}) / n
        """
        return prices.rolling(window=period).mean()

    @staticmethod
    def ema(prices: pd.Series, period: int) -> pd.Series:
        """
        Exponential Moving Average
        Formula: EMA_t = α·P_t + (1-α)·EMA_{t-1},  α = 2/(period+1)
        """
        return prices.ewm(span=period, adjust=False).mean()

    @staticmethod
    def rsi(prices: pd.Series, period: int = 14) -> pd.Series:
        """
        Relative Strength Index
        Formula: RSI = 100 - 100/(1 + AvgGain/AvgLoss)
        """
        delta    = prices.diff()
        gain     = delta.where(delta > 0, 0).rolling(period).mean()
        loss     = (-delta.where(delta < 0, 0)).rolling(period).mean()
        return 100 - (100 / (1 + gain / loss))

    @staticmethod
    def macd(prices: pd.Series, fast: int = 12, slow: int = 26,
             signal: int = 9) -> Dict[str, pd.Series]:
        """
        MACD (Moving Average Convergence Divergence)
        MACD Line  = EMA(fast) - EMA(slow)
        Signal     = EMA(9) of MACD Line
        Histogram  = MACD Line - Signal
        """
        ema_fast    = prices.ewm(span=fast,   adjust=False).mean()
        ema_slow    = prices.ewm(span=slow,   adjust=False).mean()
        macd_line   = ema_fast - ema_slow
        signal_line = macd_line.ewm(span=signal, adjust=False).mean()
        return {
            'macd':      macd_line,
            'signal':    signal_line,
            'histogram': macd_line - signal_line,
        }

    @staticmethod
    def bollinger_bands(prices: pd.Series, period: int = 20,
                        num_std: float = 2.0) -> Dict[str, pd.Series]:
        """
        Bollinger Bands
        Middle = SMA(period)
        Upper  = Middle + num_std·σ
        Lower  = Middle - num_std·σ
        %B     = (Price - Lower) / (Upper - Lower)
        """
        middle    = prices.rolling(period).mean()
        std       = prices.rolling(period).std()
        upper     = middle + num_std * std
        lower     = middle - num_std * std
        bandwidth = (upper - lower) / middle
        percent_b = (prices - lower) / (upper - lower)
        return {
            'upper':     upper,
            'middle':    middle,
            'lower':     lower,
            'bandwidth': bandwidth,
            'percent_b': percent_b,
        }

    @staticmethod
    def stochastic_oscillator(high: pd.Series, low: pd.Series,
                               close: pd.Series, period: int = 14) -> Dict:
        """
        Stochastic Oscillator
        %K = (Close - Low_n) / (High_n - Low_n) × 100
        %D = SMA(3) of %K
        """
        low_min  = low.rolling(period).min()
        high_max = high.rolling(period).max()
        k        = 100 * (close - low_min) / (high_max - low_min)
        return {'k_percent': k, 'd_percent': k.rolling(3).mean()}

    @staticmethod
    def atr(high: pd.Series, low: pd.Series, close: pd.Series,
            period: int = 14) -> pd.Series:
        """
        Average True Range
        TR = max(H-L, |H-C_prev|, |L-C_prev|)
        ATR = EMA(TR, period)
        """
        prev_c  = close.shift(1)
        tr      = pd.concat([high - low, abs(high - prev_c), abs(low - prev_c)],
                             axis=1).max(axis=1)
        return tr.ewm(span=period, adjust=False).mean()


# ── Quick test ──────────────────────────────────────────────
ind = TechnicalIndicators()
print("Fetching 6 months of AAPL data for indicator testing...")
df_test  = yf.Ticker('AAPL').history(period='6mo')
prices_s = df_test['Close']

rsi_vals = ind.rsi(prices_s)
macd_d   = ind.macd(prices_s)
bb_d     = ind.bollinger_bands(prices_s)

print(f"\nLatest Indicator Values:")
print(f"  RSI(14):          {rsi_vals.iloc[-1]:.2f}")
print(f"  MACD Line:        {macd_d['macd'].iloc[-1]:.4f}")
print(f"  MACD Signal:      {macd_d['signal'].iloc[-1]:.4f}")
print(f"  MACD Histogram:   {macd_d['histogram'].iloc[-1]:.4f}")
print(f"  BB Upper:        ${bb_d['upper'].iloc[-1]:.2f}")
print(f"  BB Middle:       ${bb_d['middle'].iloc[-1]:.2f}")
print(f"  BB Lower:        ${bb_d['lower'].iloc[-1]:.2f}")
print(f"  BB %B:            {bb_d['percent_b'].iloc[-1]:.3f}")

---
## 🎯 Section 5: Signal Generator
### `agents/alpha_signal/signal_generator.py`

In [ ]:
# ============================================================
# agents/alpha_signal/signal_generator.py
# ============================================================

class SignalGenerator:
    """
    Combines multiple technical indicators into a single
    trading signal with a weighted confidence score.
    """

    def __init__(self):
        self.ind = TechnicalIndicators()
        # Weights must sum to 1.0
        self.weights = {
            'ma_crossover': 0.25,
            'rsi':          0.25,
            'macd':         0.30,
            'bollinger':    0.20,
        }

    def _ma_crossover_signal(self, prices: pd.Series) -> Tuple[float, str]:
        sma20  = self.ind.sma(prices, 20).iloc[-1]
        sma50  = self.ind.sma(prices, 50).iloc[-1]
        price  = prices.iloc[-1]
        if pd.isna(sma20) or pd.isna(sma50):
            return 0, 'Insufficient data'
        if price > sma20 > sma50:
            return 1,  f'Bullish: {price:.2f} > SMA20({sma20:.2f}) > SMA50({sma50:.2f})'
        if price < sma20 < sma50:
            return -1, f'Bearish: {price:.2f} < SMA20({sma20:.2f}) < SMA50({sma50:.2f})'
        return 0, 'Mixed moving average signals'

    def _rsi_signal(self, prices: pd.Series) -> Tuple[float, str]:
        rsi_val = self.ind.rsi(prices).iloc[-1]
        if pd.isna(rsi_val): return 0, 'Insufficient data'
        if rsi_val < 30:  return  1,    f'Oversold: RSI={rsi_val:.1f}'
        if rsi_val > 70:  return -1,    f'Overbought: RSI={rsi_val:.1f}'
        if rsi_val < 50:  return  0.5,  f'Slightly oversold: RSI={rsi_val:.1f}'
        return -0.5, f'Slightly overbought: RSI={rsi_val:.1f}'

    def _macd_signal(self, prices: pd.Series) -> Tuple[float, str]:
        md   = self.ind.macd(prices)
        macd = md['macd'].iloc[-1]
        sig  = md['signal'].iloc[-1]
        hist = md['histogram'].iloc[-1]
        if pd.isna(macd) or pd.isna(sig): return 0, 'Insufficient data'
        if macd > sig:
            return 1,  f'Bullish MACD: {macd:.4f} > Signal {sig:.4f}'
        return -1,     f'Bearish MACD: {macd:.4f} < Signal {sig:.4f}'

    def _bollinger_signal(self, prices: pd.Series) -> Tuple[float, str]:
        bb    = self.ind.bollinger_bands(prices)
        price = prices.iloc[-1]
        upper = bb['upper'].iloc[-1]
        lower = bb['lower'].iloc[-1]
        pb    = bb['percent_b'].iloc[-1]
        if pd.isna(upper) or pd.isna(lower): return 0, 'Insufficient data'
        if price < lower: return  1,  f'Oversold: price below lower band'
        if price > upper: return -1,  f'Overbought: price above upper band'
        if pb < 0.2:      return  0.5, f'Near lower band: %B={pb:.2f}'
        if pb > 0.8:      return -0.5, f'Near upper band: %B={pb:.2f}'
        return 0, f'Mid-range: %B={pb:.2f}'

    def generate_signal(self, prices: pd.Series) -> Dict:
        """Combine all indicators into a single Buy/Sell/Hold signal."""
        signals = {
            'ma_crossover': self._ma_crossover_signal(prices),
            'rsi':          self._rsi_signal(prices),
            'macd':         self._macd_signal(prices),
            'bollinger':    self._bollinger_signal(prices),
        }
        raw_signals    = {k: v[0] for k, v in signals.items()}
        explanations   = {k: v[1] for k, v in signals.items()}
        confidence     = sum(raw_signals[k] * self.weights[k]
                             for k in self.weights) * 100
        if   confidence > 60:  rec, action = 'STRONG BUY',  'BUY'
        elif confidence > 20:  rec, action = 'BUY',          'BUY'
        elif confidence > -20: rec, action = 'HOLD',         'HOLD'
        elif confidence > -60: rec, action = 'SELL',         'SELL'
        else:                  rec, action = 'STRONG SELL',  'SELL'
        return {
            'timestamp':      datetime.now().isoformat(),
            'current_price':  prices.iloc[-1],
            'recommendation': rec,
            'action':         action,
            'confidence':     confidence,
            'signals':        raw_signals,
            'explanations':   explanations,
        }

    def backtest_strategy(self, prices: pd.Series,
                          initial_capital: float = 10_000) -> Dict:
        """Simple daily-signal backtest vs buy-and-hold."""
        if len(prices) < 50:
            return {'error': 'Insufficient data for backtest (need ≥50 days)'}
        capital, position = initial_capital, 0
        trades, equity_curve = [], []
        for i in range(50, len(prices)):
            sig   = self.generate_signal(prices.iloc[:i+1])
            price = prices.iloc[i]
            if sig['action'] == 'BUY' and position == 0 and capital > price:
                shares    = int(capital / price)
                capital  -= shares * price
                position  = shares
                trades.append({'date': prices.index[i], 'action': 'BUY',
                               'shares': shares, 'price': price})
            elif sig['action'] == 'SELL' and position > 0:
                capital  += position * price
                trades.append({'date': prices.index[i], 'action': 'SELL',
                               'shares': position, 'price': price})
                position  = 0
            equity_curve.append({'date': prices.index[i],
                                  'value': capital + position * price})
        final_val  = capital + position * prices.iloc[-1]
        total_ret  = (final_val / initial_capital - 1) * 100
        bh_shares  = initial_capital / prices.iloc[50]
        bh_return  = (bh_shares * prices.iloc[-1] / initial_capital - 1) * 100
        return {
            'initial_capital':      initial_capital,
            'final_value':          final_val,
            'total_return_pct':     total_ret,
            'num_trades':           len(trades),
            'buy_hold_return_pct':  bh_return,
            'outperformance':       total_ret - bh_return,
            'equity_curve':         equity_curve,
        }


sig_gen = SignalGenerator()
signal  = sig_gen.generate_signal(prices_s)

print(f"\n{'='*50}")
print(f"  AAPL Trading Signal")
print(f"{'='*50}")
print(f"  Price:          ${signal['current_price']:.2f}")
print(f"  Recommendation: {signal['recommendation']}")
print(f"  Action:         {signal['action']}")
print(f"  Confidence:     {signal['confidence']:.1f}%")
print(f"\n  Individual signals:")
for k, v in signal['signals'].items():
    emoji = '🟢' if v > 0 else '🔴' if v < 0 else '⚪'
    print(f"    {emoji} {k}: {signal['explanations'][k]}")
print(f"{'='*50}")

---
## ⭐ Section 6: Portfolio Optimization Agent (Week 5 Core)
### `agents/alpha_signal/agent.py`

Implements **Modern Portfolio Theory (Markowitz)**:
- Portfolio Return:  `R_p = w^T · μ`
- Portfolio Variance: `σ²_p = w^T · Σ · w`
- Sharpe Ratio: `S = (R_p - R_f) / σ_p`
- Optimization via **SciPy SLSQP**

In [ ]:
# ============================================================
# agents/alpha_signal/agent.py  —  PortfolioOptimizationAgent
# ============================================================

class PortfolioOptimizationAgent:
    """
    Portfolio Optimization Agent — Week 5
    Implements Modern Portfolio Theory (Markowitz Mean-Variance)

    Strategies:
      1. Maximum Sharpe Ratio
      2. Minimum Variance
      3. Target Return
      4. Efficient Frontier (50 optimal portfolios)
      5. Signal-Adjusted (integrates Week 4 alpha signals)
    """

    def __init__(self,
                 market_data_api: str = 'http://localhost:8000',
                 risk_agent_api:  str = 'http://localhost:8001',
                 alpha_agent_api: str = 'http://localhost:8002'):
        self.market_data_api = market_data_api
        self.risk_agent_api  = risk_agent_api
        self.alpha_agent_api = alpha_agent_api
        self.risk_free_rate  = 0.04   # 4% annual
        self._sig_gen        = SignalGenerator()

    # ── Data Fetching ────────────────────────────────────────
    def fetch_returns_data(self, symbols: List[str],
                           period: str = '1y') -> Optional[pd.DataFrame]:
        """
        Fetch daily returns for all symbols.
        Tries REST API first, falls back to direct yfinance.
        Returns a DataFrame: columns = symbols, rows = dates.
        """
        returns_dict = {}
        for sym in symbols:
            fetched = False
            # Try the Market Data API
            try:
                r = requests.get(
                    f"{self.market_data_api}/historical/{sym}",
                    params={'period': period}, timeout=5
                )
                if r.status_code == 200:
                    df = pd.DataFrame(r.json())
                    df['Date'] = pd.to_datetime(df['Date'])
                    df = df.set_index('Date')
                    returns_dict[sym] = df['Close'].pct_change().dropna()
                    fetched = True
            except Exception:
                pass
            # Fallback: direct yfinance
            if not fetched:
                try:
                    df = yf.Ticker(sym).history(period=period)
                    returns_dict[sym] = df['Close'].pct_change().dropna()
                except Exception as e:
                    print(f"  Could not fetch {sym}: {e}")
                    return None

        df_returns = pd.DataFrame(returns_dict).dropna()
        print(f"  ✓ Returns fetched: {len(df_returns)} days × {len(symbols)} symbols")
        return df_returns

    # ── Core Metrics ─────────────────────────────────────────
    def calculate_portfolio_metrics(self, weights: np.ndarray,
                                    mean_returns: np.ndarray,
                                    cov_matrix: np.ndarray
                                    ) -> Tuple[float, float, float]:
        """
        Compute annualised portfolio return, volatility, and Sharpe.

        R_p   = w^T · μ · 252
        σ_p   = sqrt(w^T · Σ_annual · w)
        Sharpe = (R_p - R_f) / σ_p
        """
        port_return  = float(np.sum(weights * mean_returns) * 252)
        port_var     = float(np.dot(weights.T, np.dot(cov_matrix * 252, weights)))
        port_vol     = float(np.sqrt(port_var))
        sharpe       = (port_return - self.risk_free_rate) / port_vol if port_vol > 0 else 0.0
        return port_return, port_vol, sharpe

    def _negative_sharpe(self, weights, mean_returns, cov_matrix):
        """Objective: minimise negative Sharpe (= maximise Sharpe)."""
        _, _, sharpe = self.calculate_portfolio_metrics(weights, mean_returns, cov_matrix)
        return -sharpe

    def _portfolio_variance(self, weights, cov_matrix):
        """Objective: minimise portfolio variance."""
        return float(np.dot(weights.T, np.dot(cov_matrix * 252, weights)))

    def _build_constraints_bounds(self, n, max_w, min_w, extra=None):
        constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
        if extra:
            constraints.extend(extra)
        bounds = tuple((min_w, max_w) for _ in range(n))
        return constraints, bounds

    # ── Strategy 1: Maximum Sharpe ───────────────────────────
    def optimize_max_sharpe(self, returns_df: pd.DataFrame,
                             max_weight: float = 0.4,
                             min_weight: float = 0.0) -> Dict:
        """
        Find weights that MAXIMISE the Sharpe ratio.

        Solver: SciPy SLSQP
        Constraints:
          Σ w_i = 1
          min_weight ≤ w_i ≤ max_weight
        """
        n            = len(returns_df.columns)
        mean_ret     = returns_df.mean().values
        cov          = returns_df.cov().values
        init_w       = np.full(n, 1.0 / n)
        cons, bounds = self._build_constraints_bounds(n, max_weight, min_weight)

        result = minimize(
            self._negative_sharpe, init_w,
            args=(mean_ret, cov),
            method='SLSQP', bounds=bounds, constraints=cons,
            options={'maxiter': 1000, 'ftol': 1e-9}
        )
        w = result.x
        ret, vol, sharpe = self.calculate_portfolio_metrics(w, mean_ret, cov)
        return {
            'weights':              dict(zip(returns_df.columns, w)),
            'expected_return':      ret,
            'volatility':           vol,
            'sharpe_ratio':         sharpe,
            'optimization_success': result.success,
            'strategy':             'Maximum Sharpe Ratio',
        }

    # ── Strategy 2: Minimum Variance ─────────────────────────
    def optimize_min_variance(self, returns_df: pd.DataFrame,
                               max_weight: float = 0.4,
                               min_weight: float = 0.0) -> Dict:
        """
        Find weights that MINIMISE portfolio variance.
        Useful for risk-averse / conservative investors.
        """
        n            = len(returns_df.columns)
        mean_ret     = returns_df.mean().values
        cov          = returns_df.cov().values
        init_w       = np.full(n, 1.0 / n)
        cons, bounds = self._build_constraints_bounds(n, max_weight, min_weight)

        result = minimize(
            self._portfolio_variance, init_w,
            args=(cov,),
            method='SLSQP', bounds=bounds, constraints=cons,
            options={'maxiter': 1000, 'ftol': 1e-9}
        )
        w = result.x
        ret, vol, sharpe = self.calculate_portfolio_metrics(w, mean_ret, cov)
        return {
            'weights':              dict(zip(returns_df.columns, w)),
            'expected_return':      ret,
            'volatility':           vol,
            'sharpe_ratio':         sharpe,
            'optimization_success': result.success,
            'strategy':             'Minimum Variance',
        }

    # ── Strategy 3: Target Return ─────────────────────────────
    def optimize_target_return(self, returns_df: pd.DataFrame,
                                target_return: float,
                                max_weight: float = 0.4,
                                min_weight: float = 0.0) -> Dict:
        """
        Minimise variance subject to an expected return ≥ target.
        """
        n        = len(returns_df.columns)
        mean_ret = returns_df.mean().values
        cov      = returns_df.cov().values
        init_w   = np.full(n, 1.0 / n)
        extra    = [{'type': 'ineq',
                     'fun': lambda w: np.sum(w * mean_ret) * 252 - target_return}]
        cons, bounds = self._build_constraints_bounds(n, max_weight, min_weight, extra)

        result = minimize(
            self._portfolio_variance, init_w,
            args=(cov,),
            method='SLSQP', bounds=bounds, constraints=cons,
            options={'maxiter': 1000, 'ftol': 1e-9}
        )
        w = result.x
        ret, vol, sharpe = self.calculate_portfolio_metrics(w, mean_ret, cov)
        return {
            'weights':              dict(zip(returns_df.columns, w)),
            'expected_return':      ret,
            'volatility':           vol,
            'sharpe_ratio':         sharpe,
            'target_return':        target_return,
            'optimization_success': result.success,
            'strategy':             f'Target Return {target_return:.1%}',
        }

    # ── Strategy 4: Efficient Frontier ───────────────────────
    def generate_efficient_frontier(self, returns_df: pd.DataFrame,
                                     num_portfolios: int = 50,
                                     max_weight: float = 0.4
                                     ) -> pd.DataFrame:
        """
        Generate `num_portfolios` optimal portfolios spanning the
        range of achievable returns — the Markowitz Efficient Frontier.
        """
        mean_ret    = returns_df.mean().values
        min_ret     = mean_ret.min() * 252
        max_ret     = mean_ret.max() * 252
        target_rets = np.linspace(min_ret, max_ret, num_portfolios)
        rows        = []
        print(f"  Building efficient frontier ({num_portfolios} portfolios) ...", end="")
        for i, tr in enumerate(target_rets):
            try:
                res = self.optimize_target_return(returns_df, tr, max_weight)
                if res['optimization_success']:
                    row = {'return': res['expected_return'],
                           'volatility': res['volatility'],
                           'sharpe': res['sharpe_ratio']}
                    row.update(res['weights'])
                    rows.append(row)
            except Exception:
                pass
            if i % 10 == 9: print(".", end="", flush=True)
        print(f" done ({len(rows)} points)")
        return pd.DataFrame(rows)

    # ── Strategy 5: Signal-Adjusted ───────────────────────────
    def optimize_with_signals(self, symbols: List[str],
                               portfolio_value: float = 100_000,
                               max_weight: float = 0.4) -> Dict:
        """
        Optimize portfolio incorporating Week 4 alpha signals.
        Adjusts base Sharpe-optimal weights by signal confidence.
        """
        returns_df = self.fetch_returns_data(symbols)
        if returns_df is None:
            return {'status': 'error', 'message': 'Could not fetch returns'}

        # Get signals (fallback: generate directly from yfinance)
        signals = {}
        for sym in symbols:
            try:
                r = requests.get(f"{self.alpha_agent_api}/signal/{sym}",
                                 params={'portfolio_value': portfolio_value},
                                 timeout=5)
                if r.status_code == 200:
                    sd = r.json()
                    signals[sym] = sd.get('confidence', 0) / 100
                    continue
            except Exception:
                pass
            # Direct fallback
            try:
                df_sym  = yf.Ticker(sym).history(period='6mo')
                sg      = self._sig_gen.generate_signal(df_sym['Close'])
                signals[sym] = sg['confidence'] / 100
            except Exception:
                signals[sym] = 0.0

        base     = self.optimize_max_sharpe(returns_df, max_weight)
        adj_w    = {}
        for sym in symbols:
            bw  = base['weights'][sym]
            adj = signals.get(sym, 0) * 0.2 * bw   # ±20% adjustment
            adj_w[sym] = max(0.0, min(max_weight, bw + adj))

        total  = sum(adj_w.values())
        adj_w  = {k: v / total for k, v in adj_w.items()}

        mean_ret   = returns_df.mean().values
        cov        = returns_df.cov().values
        w_arr      = np.array([adj_w[s] for s in symbols])
        ret, vol, sharpe = self.calculate_portfolio_metrics(w_arr, mean_ret, cov)

        return {
            'status':           'success',
            'strategy':         'Signal-Adjusted Max Sharpe',
            'base_weights':     base['weights'],
            'adjusted_weights': adj_w,
            'signals':          signals,
            'expected_return':  ret,
            'volatility':       vol,
            'sharpe_ratio':     sharpe,
            'portfolio_value':  portfolio_value,
            'allocation_usd':   {s: adj_w[s] * portfolio_value for s in symbols},
        }


print("PortfolioOptimizationAgent defined ✅")

---
## 🧪 Section 7: Optimization Demo — All Strategies

In [ ]:
# ── Initialize agent & fetch data ───────────────────────────
portfolio_agent = PortfolioOptimizationAgent()

print("Fetching returns for:", SYMBOLS)
returns_df = portfolio_agent.fetch_returns_data(SYMBOLS, period='1y')
print(f"Shape: {returns_df.shape}  |  Date range: {returns_df.index[0].date()} → {returns_df.index[-1].date()}")

In [ ]:
# ── Strategy 1: Maximum Sharpe ───────────────────────────────
print("\n" + "="*60)
print("  Strategy 1: MAXIMUM SHARPE RATIO")
print("="*60)

max_sharpe = portfolio_agent.optimize_max_sharpe(returns_df, max_weight=0.4)

print(f"  Status:          {'✅ Success' if max_sharpe['optimization_success'] else '⚠ Warning'}")
print(f"  Expected Return: {max_sharpe['expected_return']*100:.2f}%")
print(f"  Volatility:      {max_sharpe['volatility']*100:.2f}%")
print(f"  Sharpe Ratio:    {max_sharpe['sharpe_ratio']:.3f}")
print(f"\n  Optimal Weights:")
for sym, w in sorted(max_sharpe['weights'].items(), key=lambda x: -x[1]):
    bar = '█' * int(w * 30)
    print(f"    {sym:6s}: {w*100:6.2f}%  {bar}")

In [ ]:
# ── Strategy 2: Minimum Variance ─────────────────────────────
print("\n" + "="*60)
print("  Strategy 2: MINIMUM VARIANCE")
print("="*60)

min_var = portfolio_agent.optimize_min_variance(returns_df, max_weight=0.4)

print(f"  Expected Return: {min_var['expected_return']*100:.2f}%")
print(f"  Volatility:      {min_var['volatility']*100:.2f}%")
print(f"  Sharpe Ratio:    {min_var['sharpe_ratio']:.3f}")
print(f"\n  Optimal Weights:")
for sym, w in sorted(min_var['weights'].items(), key=lambda x: -x[1]):
    bar = '█' * int(w * 30)
    print(f"    {sym:6s}: {w*100:6.2f}%  {bar}")

In [ ]:
# ── Strategy 3: Target Return ─────────────────────────────────
print("\n" + "="*60)
print("  Strategy 3: TARGET RETURN (15%)")
print("="*60)

target = portfolio_agent.optimize_target_return(returns_df, target_return=0.15, max_weight=0.4)

print(f"  Target Return:   {target['target_return']*100:.1f}%")
print(f"  Achieved Return: {target['expected_return']*100:.2f}%")
print(f"  Volatility:      {target['volatility']*100:.2f}%")
print(f"  Sharpe Ratio:    {target['sharpe_ratio']:.3f}")
print(f"\n  Optimal Weights:")
for sym, w in sorted(target['weights'].items(), key=lambda x: -x[1]):
    bar = '█' * int(w * 30)
    print(f"    {sym:6s}: {w*100:6.2f}%  {bar}")

In [ ]:
# ── Strategy 4: Signal-Adjusted ───────────────────────────────
print("\n" + "="*60)
print("  Strategy 5: SIGNAL-ADJUSTED (Week 4 Integration)")
print("="*60)

sig_adj = portfolio_agent.optimize_with_signals(SYMBOLS, portfolio_value=100_000)

if sig_adj['status'] == 'success':
    print(f"  Expected Return: {sig_adj['expected_return']*100:.2f}%")
    print(f"  Volatility:      {sig_adj['volatility']*100:.2f}%")
    print(f"  Sharpe Ratio:    {sig_adj['sharpe_ratio']:.3f}")
    print(f"\n  Alpha Signals:")
    for sym, sig in sig_adj['signals'].items():
        emoji = '🟢' if sig > 0 else '🔴' if sig < 0 else '⚪'
        print(f"    {emoji} {sym}: {sig*100:+.1f}%")
    print(f"\n  Allocation (${sig_adj['portfolio_value']:,.0f} portfolio):")
    for sym, usd in sorted(sig_adj['allocation_usd'].items(), key=lambda x: -x[1]):
        w = sig_adj['adjusted_weights'][sym]
        bw = sig_adj['base_weights'][sym]
        diff = (w - bw) * 100
        print(f"    {sym:6s}: ${usd:>10,.0f}  ({w*100:.1f}%)  [base {bw*100:.1f}%, adj {diff:+.1f}%]")

---
## 📈 Section 8: Efficient Frontier Visualization

In [ ]:
# Generate efficient frontier
frontier_df = portfolio_agent.generate_efficient_frontier(
    returns_df, num_portfolios=50, max_weight=0.4
)
print(f"\nFrontier has {len(frontier_df)} portfolios")
frontier_df[['return', 'volatility', 'sharpe']].describe().round(4)

In [ ]:
# ── Plot Efficient Frontier ─────────────────────────────────
BLUE   = '#1a5fca'
GREEN  = '#0d9c5b'
RED    = '#e03131'
GOLD   = '#d4940a'

max_s_idx  = frontier_df['sharpe'].idxmax()
min_v_idx  = frontier_df['volatility'].idxmin()
ms_point   = frontier_df.loc[max_s_idx]
mv_point   = frontier_df.loc[min_v_idx]

fig = go.Figure()

# Frontier curve coloured by Sharpe
fig.add_trace(go.Scatter(
    x=frontier_df['volatility'] * 100,
    y=frontier_df['return']     * 100,
    mode='markers+lines',
    marker=dict(
        color=frontier_df['sharpe'],
        colorscale='Blues',
        size=8,
        showscale=True,
        colorbar=dict(title='Sharpe Ratio'),
        line=dict(color='white', width=0.5),
    ),
    line=dict(color='rgba(26,95,202,0.3)', width=1.5),
    name='Efficient Frontier',
    hovertemplate=(
        '<b>Efficient Portfolio</b><br>'
        'Return: %{y:.2f}%<br>'
        'Volatility: %{x:.2f}%<br>'
        'Sharpe: %{marker.color:.3f}<extra></extra>'
    )
))

# Max Sharpe point
fig.add_trace(go.Scatter(
    x=[ms_point['volatility'] * 100],
    y=[ms_point['return']     * 100],
    mode='markers+text',
    marker=dict(color=GOLD, size=16, symbol='star',
                line=dict(color='white', width=2)),
    text=['⭐ Max Sharpe'],
    textposition='top center',
    textfont=dict(color=GOLD, size=11),
    name=f'Max Sharpe ({ms_point["sharpe"]:.3f})',
    hovertemplate=(
        '<b>Maximum Sharpe Portfolio</b><br>'
        f'Return: {ms_point["return"]*100:.2f}%<br>'
        f'Volatility: {ms_point["volatility"]*100:.2f}%<br>'
        f'Sharpe: {ms_point["sharpe"]:.3f}<extra></extra>'
    )
))

# Min Variance point
fig.add_trace(go.Scatter(
    x=[mv_point['volatility'] * 100],
    y=[mv_point['return']     * 100],
    mode='markers+text',
    marker=dict(color=GREEN, size=14, symbol='diamond',
                line=dict(color='white', width=2)),
    text=['🛡 Min Variance'],
    textposition='bottom center',
    textfont=dict(color=GREEN, size=11),
    name=f'Min Variance ({mv_point["volatility"]*100:.2f}%)',
    hovertemplate=(
        '<b>Minimum Variance Portfolio</b><br>'
        f'Return: {mv_point["return"]*100:.2f}%<br>'
        f'Volatility: {mv_point["volatility"]*100:.2f}%<br>'
        f'Sharpe: {mv_point["sharpe"]:.3f}<extra></extra>'
    )
))

fig.update_layout(
    title=dict(
        text='⬡ Markowitz Efficient Frontier — AAPL · GOOGL · MSFT · TSLA · AMZN',
        font=dict(size=16, color='#0d2d6b', family='Arial')
    ),
    xaxis=dict(title='Annualised Volatility (%)', gridcolor='#e2eaf5'),
    yaxis=dict(title='Annualised Return (%)',     gridcolor='#e2eaf5'),
    paper_bgcolor='white',
    plot_bgcolor='#f8faff',
    legend=dict(bgcolor='rgba(255,255,255,0.9)', bordercolor='#d0dff0',
                borderwidth=1),
    height=520,
    margin=dict(l=60, r=30, t=70, b=60),
    hovermode='closest',
)

fig.show()
print(f"\nMax Sharpe Portfolio:  Return={ms_point['return']*100:.2f}%  Vol={ms_point['volatility']*100:.2f}%  Sharpe={ms_point['sharpe']:.3f}")
print(f"Min Variance Portfolio: Return={mv_point['return']*100:.2f}%  Vol={mv_point['volatility']*100:.2f}%  Sharpe={mv_point['sharpe']:.3f}")

In [ ]:
# ── Strategy Comparison Table ────────────────────────────────
comparison_data = [
    {
        'Strategy':        'Max Sharpe',
        'Expected Return': f"{max_sharpe['expected_return']*100:.2f}%",
        'Volatility':      f"{max_sharpe['volatility']*100:.2f}%",
        'Sharpe Ratio':    f"{max_sharpe['sharpe_ratio']:.3f}",
        **{s: f"{w*100:.1f}%" for s, w in max_sharpe['weights'].items()}
    },
    {
        'Strategy':        'Min Variance',
        'Expected Return': f"{min_var['expected_return']*100:.2f}%",
        'Volatility':      f"{min_var['volatility']*100:.2f}%",
        'Sharpe Ratio':    f"{min_var['sharpe_ratio']:.3f}",
        **{s: f"{w*100:.1f}%" for s, w in min_var['weights'].items()}
    },
    {
        'Strategy':        'Target 15%',
        'Expected Return': f"{target['expected_return']*100:.2f}%",
        'Volatility':      f"{target['volatility']*100:.2f}%",
        'Sharpe Ratio':    f"{target['sharpe_ratio']:.3f}",
        **{s: f"{w*100:.1f}%" for s, w in target['weights'].items()}
    },
]
if sig_adj['status'] == 'success':
    comparison_data.append({
        'Strategy':        'Signal-Adjusted',
        'Expected Return': f"{sig_adj['expected_return']*100:.2f}%",
        'Volatility':      f"{sig_adj['volatility']*100:.2f}%",
        'Sharpe Ratio':    f"{sig_adj['sharpe_ratio']:.3f}",
        **{s: f"{w*100:.1f}%" for s, w in sig_adj['adjusted_weights'].items()}
    })

cmp_df = pd.DataFrame(comparison_data).set_index('Strategy')
print("\n📊 Strategy Comparison:")
cmp_df

In [ ]:
# ── Portfolio Weight Pie Charts ───────────────────────────────
fig2 = make_subplots(
    rows=1, cols=2, specs=[[{'type': 'pie'}, {'type': 'pie'}]],
    subplot_titles=['Maximum Sharpe Portfolio', 'Minimum Variance Portfolio']
)
PALETTE = ['#1a5fca','#0d9c5b','#d4940a','#e03131','#7c3aed']

fig2.add_trace(go.Pie(
    labels=list(max_sharpe['weights'].keys()),
    values=[round(v*100, 2) for v in max_sharpe['weights'].values()],
    marker=dict(colors=PALETTE, line=dict(color='white', width=2)),
    textinfo='label+percent', hole=0.35,
    hovertemplate='%{label}: %{value:.2f}%<extra></extra>'
), row=1, col=1)

fig2.add_trace(go.Pie(
    labels=list(min_var['weights'].keys()),
    values=[round(v*100, 2) for v in min_var['weights'].values()],
    marker=dict(colors=PALETTE, line=dict(color='white', width=2)),
    textinfo='label+percent', hole=0.35,
    hovertemplate='%{label}: %{value:.2f}%<extra></extra>'
), row=1, col=2)

fig2.update_layout(
    title=dict(text='Portfolio Weight Allocation', font=dict(size=15, color='#0d2d6b')),
    paper_bgcolor='white',
    height=420,
    showlegend=False,
)
fig2.show()

---
## 🔌 Section 9: REST API Definition
### `agents/alpha_signal/api.py`

The code below defines the FastAPI app. Run it in a **separate terminal** with:
```bash
cd agents/alpha_signal
python api.py
```

In [ ]:
# ============================================================
# agents/alpha_signal/api.py
# Portfolio Optimization REST API  —  Port 8003
# ============================================================
API_CODE = '''
from fastapi import FastAPI, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List
import uvicorn
from agent import PortfolioOptimizationAgent

app = FastAPI(
    title="Portfolio Optimization Agent API",
    description="Modern Portfolio Theory optimization using Markowitz Mean-Variance",
    version="1.0.0"
)
app.add_middleware(
    CORSMiddleware, allow_origins=["*"],
    allow_methods=["*"], allow_headers=["*"]
)
agent = PortfolioOptimizationAgent()


class OptimizeRequest(BaseModel):
    symbols:    List[str]
    max_weight: float = 0.4
    min_weight: float = 0.0
    period:     str   = "1y"


class TargetReturnRequest(BaseModel):
    symbols:       List[str]
    target_return: float
    max_weight:    float = 0.4
    min_weight:    float = 0.0
    period:        str   = "1y"


class SignalOptimizeRequest(BaseModel):
    symbols:          List[str]
    portfolio_value:  float = 100_000
    max_weight:       float = 0.4
    period:           str   = "1y"


@app.get("/")
def root():
    return {
        "service": "Portfolio Optimization Agent",
        "status":  "running",
        "version": "1.0.0",
        "endpoints": [
            "POST /optimize/max-sharpe",
            "POST /optimize/min-variance",
            "POST /optimize/target-return",
            "POST /optimize/with-signals",
            "POST /efficient-frontier",
            "GET  /optimize/{type}?symbols=AAPL,GOOGL,MSFT",
        ]
    }


@app.post("/optimize/max-sharpe")
def optimize_max_sharpe(req: OptimizeRequest):
    returns_df = agent.fetch_returns_data(req.symbols, req.period)
    if returns_df is None:
        raise HTTPException(400, "Could not fetch returns data")
    result = agent.optimize_max_sharpe(returns_df, req.max_weight, req.min_weight)
    result.update(symbols=req.symbols, period=req.period, status="success")
    return result


@app.post("/optimize/min-variance")
def optimize_min_variance(req: OptimizeRequest):
    returns_df = agent.fetch_returns_data(req.symbols, req.period)
    if returns_df is None:
        raise HTTPException(400, "Could not fetch returns data")
    result = agent.optimize_min_variance(returns_df, req.max_weight, req.min_weight)
    result.update(symbols=req.symbols, period=req.period, status="success")
    return result


@app.post("/optimize/target-return")
def optimize_target_return(req: TargetReturnRequest):
    returns_df = agent.fetch_returns_data(req.symbols, req.period)
    if returns_df is None:
        raise HTTPException(400, "Could not fetch returns data")
    result = agent.optimize_target_return(
        returns_df, req.target_return, req.max_weight, req.min_weight
    )
    result.update(symbols=req.symbols, period=req.period, status="success")
    return result


@app.post("/optimize/with-signals")
def optimize_with_signals(req: SignalOptimizeRequest):
    result = agent.optimize_with_signals(
        req.symbols, req.portfolio_value, req.max_weight
    )
    if result.get("status") == "error":
        raise HTTPException(400, result["message"])
    result["period"] = req.period
    return result


@app.post("/efficient-frontier")
def generate_efficient_frontier(
    req: OptimizeRequest,
    num_portfolios: int = Query(50, description="Number of frontier portfolios")
):
    returns_df = agent.fetch_returns_data(req.symbols, req.period)
    if returns_df is None:
        raise HTTPException(400, "Could not fetch returns data")
    df = agent.generate_efficient_frontier(returns_df, num_portfolios, req.max_weight)
    return {
        "status":         "success",
        "symbols":        req.symbols,
        "num_portfolios": len(df),
        "frontier":       df.to_dict(orient="records"),
    }


@app.get("/optimize/{optimization_type}")
def quick_optimize(
    optimization_type: str,
    symbols:    str   = Query(..., description="Comma-separated symbols"),
    max_weight: float = Query(0.4),
    period:     str   = Query("1y")
):
    syms       = [s.strip().upper() for s in symbols.split(",")]
    returns_df = agent.fetch_returns_data(syms, period)
    if returns_df is None:
        raise HTTPException(400, "Could not fetch returns data")
    if optimization_type == "max-sharpe":
        result = agent.optimize_max_sharpe(returns_df, max_weight)
    elif optimization_type == "min-variance":
        result = agent.optimize_min_variance(returns_df, max_weight)
    else:
        raise HTTPException(400, f"Unknown type: {optimization_type}")
    result.update(symbols=syms, period=period, status="success")
    return result


if __name__ == "__main__":
    print("Starting Portfolio Optimization API on port 8003 ...")
    print("Swagger UI → http://localhost:8003/docs")
    uvicorn.run(app, host="0.0.0.0", port=8003)
'''

# Write the file
api_path = 'agents/alpha_signal/api.py'
os.makedirs(os.path.dirname(api_path), exist_ok=True)
with open(api_path, 'w') as f:
    f.write(API_CODE.strip())

print(f"✅ Written: {api_path}")
print("\nTo start the API:")
print("  cd agents/alpha_signal")
print("  python api.py")
print("\nSwagger UI: http://localhost:8003/docs")

---
## 🖥️ Section 10: Gradio Dashboard

Interactive Portfolio Optimization UI with 4 tabs.

In [ ]:
import gradio as gr

# ── Shared agent instance for UI ─────────────────────────────
_ui_agent = PortfolioOptimizationAgent()

PALETTE = ['#1a5fca','#0d9c5b','#d4940a','#e03131','#7c3aed']
SYMS_DEFAULT = 'AAPL,GOOGL,MSFT,TSLA,AMZN'


def _parse_symbols(s):
    return [x.strip().upper() for x in s.split(',') if x.strip()]


def _weights_html(weights: dict, portfolio_value: float) -> str:
    rows = ''
    for sym, w in sorted(weights.items(), key=lambda x: -x[1]):
        usd = w * portfolio_value
        bar = int(w * 200)
        rows += f'''
        <tr>
          <td style="font-weight:600">{sym}</td>
          <td>{w*100:.2f}%</td>
          <td>${usd:,.0f}</td>
          <td><div style="background:#1a5fca;height:12px;width:{bar}px;border-radius:4px"></div></td>
        </tr>'''
    return f'''
    <table style="width:100%;border-collapse:collapse;font-size:14px">
      <thead>
        <tr style="background:#1a3a6b;color:#fff">
          <th style="padding:8px">Symbol</th>
          <th style="padding:8px">Weight</th>
          <th style="padding:8px">Amount ($)</th>
          <th style="padding:8px">Allocation</th>
        </tr>
      </thead>
      <tbody>{rows}</tbody>
    </table>'''


def tab_max_sharpe(symbols_str, max_weight_pct, period, portfolio_value):
    syms   = _parse_symbols(symbols_str)
    max_w  = max_weight_pct / 100
    try:
        df_ret = _ui_agent.fetch_returns_data(syms, period)
        res    = _ui_agent.optimize_max_sharpe(df_ret, max_weight=max_w)
    except Exception as e:
        return f'<p style="color:red">Error: {e}</p>', None

    status   = '✅ Optimisation Successful' if res['optimization_success'] else '⚠ Converged with warnings'
    summary  = f'''
    <div style="background:#e8f0fe;border-left:4px solid #1a5fca;
                padding:14px;border-radius:8px;margin-bottom:12px">
      <h3 style="margin:0;color:#0d2d6b">⭐ Maximum Sharpe Ratio Portfolio</h3>
      <p style="margin:4px 0">{status}</p>
    </div>
    <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin-bottom:14px">
      <div style="background:#fff;border:1px solid #d0dff0;border-radius:10px;
                  padding:14px;text-align:center">
        <div style="font-size:.7rem;color:#6b83a8;text-transform:uppercase;letter-spacing:1px">Expected Return</div>
        <div style="font-size:1.6rem;font-weight:700;color:#0d9c5b">{res["expected_return"]*100:.2f}%</div>
      </div>
      <div style="background:#fff;border:1px solid #d0dff0;border-radius:10px;
                  padding:14px;text-align:center">
        <div style="font-size:.7rem;color:#6b83a8;text-transform:uppercase;letter-spacing:1px">Volatility</div>
        <div style="font-size:1.6rem;font-weight:700;color:#e03131">{res["volatility"]*100:.2f}%</div>
      </div>
      <div style="background:#fff;border:1px solid #d0dff0;border-radius:10px;
                  padding:14px;text-align:center">
        <div style="font-size:.7rem;color:#6b83a8;text-transform:uppercase;letter-spacing:1px">Sharpe Ratio</div>
        <div style="font-size:1.6rem;font-weight:700;color:#d4940a">{res["sharpe_ratio"]:.3f}</div>
      </div>
    </div>
    {_weights_html(res["weights"], portfolio_value)}
    '''

    fig = go.Figure(go.Pie(
        labels=list(res['weights'].keys()),
        values=[round(v*100, 2) for v in res['weights'].values()],
        marker=dict(colors=PALETTE, line=dict(color='white', width=2)),
        textinfo='label+percent', hole=0.38,
    ))
    fig.update_layout(
        title='Weight Allocation', paper_bgcolor='white', height=380,
        margin=dict(l=20, r=20, t=50, b=20)
    )
    return summary, fig


def tab_min_variance(symbols_str, max_weight_pct, period, portfolio_value):
    syms  = _parse_symbols(symbols_str)
    max_w = max_weight_pct / 100
    try:
        df_ret = _ui_agent.fetch_returns_data(syms, period)
        res    = _ui_agent.optimize_min_variance(df_ret, max_weight=max_w)
    except Exception as e:
        return f'<p style="color:red">Error: {e}</p>', None

    summary = f'''
    <div style="background:#d1fae5;border-left:4px solid #0d9c5b;
                padding:14px;border-radius:8px;margin-bottom:12px">
      <h3 style="margin:0;color:#065f46">🛡️ Minimum Variance Portfolio</h3>
      <p style="margin:4px 0">Conservative — lowest possible risk</p>
    </div>
    <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin-bottom:14px">
      <div style="background:#fff;border:1px solid #d0dff0;border-radius:10px;padding:14px;text-align:center">
        <div style="font-size:.7rem;color:#6b83a8;text-transform:uppercase">Expected Return</div>
        <div style="font-size:1.6rem;font-weight:700;color:#0d9c5b">{res["expected_return"]*100:.2f}%</div>
      </div>
      <div style="background:#fff;border:1px solid #d0dff0;border-radius:10px;padding:14px;text-align:center">
        <div style="font-size:.7rem;color:#6b83a8;text-transform:uppercase">Volatility</div>
        <div style="font-size:1.6rem;font-weight:700;color:#1a5fca">{res["volatility"]*100:.2f}%</div>
      </div>
      <div style="background:#fff;border:1px solid #d0dff0;border-radius:10px;padding:14px;text-align:center">
        <div style="font-size:.7rem;color:#6b83a8;text-transform:uppercase">Sharpe Ratio</div>
        <div style="font-size:1.6rem;font-weight:700;color:#d4940a">{res["sharpe_ratio"]:.3f}</div>
      </div>
    </div>
    {_weights_html(res["weights"], portfolio_value)}
    '''

    fig = go.Figure(go.Pie(
        labels=list(res['weights'].keys()),
        values=[round(v*100, 2) for v in res['weights'].values()],
        marker=dict(colors=PALETTE, line=dict(color='white', width=2)),
        textinfo='label+percent', hole=0.38,
    ))
    fig.update_layout(
        title='Weight Allocation — Min Variance', paper_bgcolor='white',
        height=380, margin=dict(l=20, r=20, t=50, b=20)
    )
    return summary, fig


def tab_frontier(symbols_str, max_weight_pct, period, n_pts):
    syms  = _parse_symbols(symbols_str)
    max_w = max_weight_pct / 100
    try:
        df_ret = _ui_agent.fetch_returns_data(syms, period)
        fdf    = _ui_agent.generate_efficient_frontier(df_ret, int(n_pts), max_w)
    except Exception as e:
        return None, f'<p style="color:red">Error: {e}</p>'

    ms_idx  = fdf['sharpe'].idxmax()
    mv_idx  = fdf['volatility'].idxmin()
    ms_pt   = fdf.loc[ms_idx]
    mv_pt   = fdf.loc[mv_idx]

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=fdf['volatility']*100, y=fdf['return']*100,
        mode='markers+lines',
        marker=dict(color=fdf['sharpe'], colorscale='Blues', size=8,
                    showscale=True, colorbar=dict(title='Sharpe'),
                    line=dict(color='white', width=0.5)),
        line=dict(color='rgba(26,95,202,0.25)', width=1),
        name='Efficient Frontier',
    ))
    fig.add_trace(go.Scatter(
        x=[ms_pt['volatility']*100], y=[ms_pt['return']*100],
        mode='markers+text', text=['⭐ Max Sharpe'], textposition='top center',
        marker=dict(color='#d4940a', size=16, symbol='star',
                    line=dict(color='white', width=2)),
        name=f'Max Sharpe ({ms_pt["sharpe"]:.3f})'
    ))
    fig.add_trace(go.Scatter(
        x=[mv_pt['volatility']*100], y=[mv_pt['return']*100],
        mode='markers+text', text=['🛡 Min Vol'], textposition='bottom center',
        marker=dict(color='#0d9c5b', size=14, symbol='diamond',
                    line=dict(color='white', width=2)),
        name=f'Min Variance'
    ))
    fig.update_layout(
        title='Markowitz Efficient Frontier',
        xaxis_title='Annualised Volatility (%)',
        yaxis_title='Annualised Return (%)',
        paper_bgcolor='white', plot_bgcolor='#f8faff',
        height=500, margin=dict(l=60, r=30, t=60, b=60)
    )

    info = f'''
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:14px;margin-top:12px">
      <div style="background:#fef3c7;border:1px solid #fcd34d;border-radius:10px;padding:14px">
        <b>⭐ Max Sharpe Portfolio</b><br>
        Return: {ms_pt["return"]*100:.2f}%  |  Vol: {ms_pt["volatility"]*100:.2f}%  |  Sharpe: {ms_pt["sharpe"]:.3f}
      </div>
      <div style="background:#d1fae5;border:1px solid #6ee7b7;border-radius:10px;padding:14px">
        <b>🛡 Min Variance Portfolio</b><br>
        Return: {mv_pt["return"]*100:.2f}%  |  Vol: {mv_pt["volatility"]*100:.2f}%  |  Sharpe: {mv_pt["sharpe"]:.3f}
      </div>
    </div>
    '''
    return fig, info


def tab_signal_adjusted(symbols_str, max_weight_pct, portfolio_value):
    syms  = _parse_symbols(symbols_str)
    max_w = max_weight_pct / 100
    try:
        res = _ui_agent.optimize_with_signals(syms, portfolio_value, max_w)
    except Exception as e:
        return f'<p style="color:red">Error: {e}</p>', None

    if res['status'] != 'success':
        return f'<p style="color:red">{res.get("message", "error")}</p>', None

    sig_rows = ''.join(
        f'<tr><td>{sym}</td><td>{"🟢" if s>0 else "🔴" if s<0 else "⚪"}</td>'
        f'<td>{s*100:+.1f}%</td></tr>'
        for sym, s in res['signals'].items()
    )

    summary = f'''
    <div style="background:#e8f0fe;border-left:4px solid #1a5fca;
                padding:14px;border-radius:8px;margin-bottom:12px">
      <h3 style="margin:0;color:#0d2d6b">🎯 Signal-Adjusted Portfolio</h3>
    </div>
    <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin-bottom:14px">
      <div style="background:#fff;border:1px solid #d0dff0;border-radius:10px;padding:14px;text-align:center">
        <div style="font-size:.7rem;color:#6b83a8;text-transform:uppercase">Expected Return</div>
        <div style="font-size:1.6rem;font-weight:700;color:#0d9c5b">{res["expected_return"]*100:.2f}%</div>
      </div>
      <div style="background:#fff;border:1px solid #d0dff0;border-radius:10px;padding:14px;text-align:center">
        <div style="font-size:.7rem;color:#6b83a8;text-transform:uppercase">Volatility</div>
        <div style="font-size:1.6rem;font-weight:700;color:#e03131">{res["volatility"]*100:.2f}%</div>
      </div>
      <div style="background:#fff;border:1px solid #d0dff0;border-radius:10px;padding:14px;text-align:center">
        <div style="font-size:.7rem;color:#6b83a8;text-transform:uppercase">Sharpe Ratio</div>
        <div style="font-size:1.6rem;font-weight:700;color:#d4940a">{res["sharpe_ratio"]:.3f}</div>
      </div>
    </div>
    <h4>Alpha Signals Used</h4>
    <table style="width:100%;border-collapse:collapse;font-size:14px">
      <thead><tr style="background:#1a3a6b;color:#fff">
        <th style="padding:8px">Symbol</th>
        <th style="padding:8px">Direction</th>
        <th style="padding:8px">Confidence</th>
      </tr></thead>
      <tbody>{sig_rows}</tbody>
    </table>
    <h4 style="margin-top:14px">Adjusted Allocation</h4>
    {_weights_html(res["adjusted_weights"], portfolio_value)}
    '''

    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'pie'}, {'type': 'pie'}]],
        subplot_titles=['Base Weights (Max Sharpe)', 'Adjusted Weights']
    )
    fig.add_trace(go.Pie(
        labels=list(res['base_weights'].keys()),
        values=[round(v*100, 2) for v in res['base_weights'].values()],
        marker=dict(colors=PALETTE, line=dict(color='white', width=2)),
        textinfo='label+percent', hole=0.35,
    ), row=1, col=1)
    fig.add_trace(go.Pie(
        labels=list(res['adjusted_weights'].keys()),
        values=[round(v*100, 2) for v in res['adjusted_weights'].values()],
        marker=dict(colors=PALETTE, line=dict(color='white', width=2)),
        textinfo='label+percent', hole=0.35,
    ), row=1, col=2)
    fig.update_layout(
        title='Base vs Signal-Adjusted Weights',
        paper_bgcolor='white', height=380, showlegend=False,
        margin=dict(l=20, r=20, t=50, b=20)
    )
    return summary, fig


# ── Build Gradio App ─────────────────────────────────────────
HEADER = """
<div style="background:linear-gradient(135deg,#0d2d6b,#1a5fca);
            padding:28px 36px;text-align:center;
            border-bottom:4px solid #d4940a;">
  <div style="font-size:1.8rem;font-weight:700;color:#fff;letter-spacing:1px">
    ⬡ Portfolio Optimization Agent — Week 5
  </div>
  <div style="font-size:.78rem;color:rgba(255,255,255,.7);
              letter-spacing:2px;text-transform:uppercase;margin-top:6px">
    Markowitz Mean-Variance Optimization &nbsp;·&nbsp; IIT Madras 2026
  </div>
  <div style="font-size:.72rem;color:rgba(255,255,255,.5);margin-top:6px;letter-spacing:1.5px">
    Team: Ashwini &nbsp;|&nbsp; Dibyendu Sarkar &nbsp;|&nbsp; Jyoti Ranjan Sethi
  </div>
</div>
"""

with gr.Blocks(title='Portfolio Optimization Agent') as demo:
    gr.HTML(HEADER)

    with gr.Row():
        g_symbols    = gr.Textbox(value=SYMS_DEFAULT, label='📌 Symbols (comma-separated)',  scale=3)
        g_period     = gr.Dropdown(['1mo','3mo','6mo','1y','2y'], value='1y',
                                   label='📅 Period',                             scale=1)
        g_max_weight = gr.Slider(10, 100, value=40, step=5,
                                 label='⚖️ Max Weight per Asset (%)',            scale=2)
        g_portfolio  = gr.Number(value=100_000, label='💰 Portfolio Value ($)',   scale=2)

    with gr.Tabs():

        with gr.Tab('⭐ Max Sharpe Ratio'):
            gr.Markdown('**Maximize risk-adjusted returns** — best Sharpe ratio portfolio.')
            ms_btn   = gr.Button('🚀 Optimize Max Sharpe', variant='primary')
            ms_html  = gr.HTML()
            ms_chart = gr.Plot(label='Allocation')
            ms_btn.click(tab_max_sharpe,
                         inputs=[g_symbols, g_max_weight, g_period, g_portfolio],
                         outputs=[ms_html, ms_chart])

        with gr.Tab('🛡️ Min Variance'):
            gr.Markdown('**Minimize portfolio risk** — ideal for conservative investors.')
            mv_btn   = gr.Button('🚀 Optimize Min Variance', variant='primary')
            mv_html  = gr.HTML()
            mv_chart = gr.Plot(label='Allocation')
            mv_btn.click(tab_min_variance,
                         inputs=[g_symbols, g_max_weight, g_period, g_portfolio],
                         outputs=[mv_html, mv_chart])

        with gr.Tab('📈 Efficient Frontier'):
            gr.Markdown('**Plot all optimal portfolios** across the risk-return spectrum.')
            ef_n_pts = gr.Slider(10, 100, value=40, step=10, label='Number of frontier points')
            ef_btn   = gr.Button('🚀 Generate Frontier', variant='primary')
            ef_chart = gr.Plot(label='Efficient Frontier')
            ef_info  = gr.HTML()
            ef_btn.click(tab_frontier,
                         inputs=[g_symbols, g_max_weight, g_period, ef_n_pts],
                         outputs=[ef_chart, ef_info])

        with gr.Tab('🎯 Signal-Adjusted'):
            gr.Markdown('**Integrate Week 4 alpha signals** to tilt base Sharpe weights.')
            sa_btn   = gr.Button('🚀 Signal-Adjusted Optimize', variant='primary')
            sa_html  = gr.HTML()
            sa_chart = gr.Plot(label='Base vs Adjusted Weights')
            sa_btn.click(tab_signal_adjusted,
                         inputs=[g_symbols, g_max_weight, g_portfolio],
                         outputs=[sa_html, sa_chart])

    gr.HTML("""
    <div style="background:#0d2d6b;color:rgba(255,255,255,.6);text-align:center;
                padding:14px;font-size:.7rem;letter-spacing:1.5px;text-transform:uppercase;margin-top:32px">
      ⬡ Portfolio Intelligence System · IIT Madras 2026 · For Educational Use Only
    </div>
    """)

print("\nLaunching Gradio dashboard ...")
print("If running in Colab/Jupyter, a public URL will be printed below.")
demo.launch(share=True, server_port=7863)

---
## 📁 Section 11: Write All Source Files to Disk

Recreates the full project file structure.

In [ ]:
import inspect, textwrap

# ── Helper ───────────────────────────────────────────────────
def write_file(path: str, content: str):
    os.makedirs(os.path.dirname(path) if '/' in path else '.', exist_ok=True)
    with open(path, 'w') as f:
        f.write(content)
    print(f"  ✓ {path}")


print("Writing project files ...\n")

# ── requirements.txt ─────────────────────────────────────────
write_file('requirements.txt', textwrap.dedent("""\
    yfinance
    pandas
    numpy
    scipy
    fastapi
    uvicorn[standard]
    pydantic
    requests
    plotly
    gradio>=4.0
    python-dateutil
    pytz
    streamlit
    python-dotenv
"""))

# ── agents/__init__.py ───────────────────────────────────────
write_file('agents/__init__.py', textwrap.dedent("""\
    \"\"\"
    Multi-Agent Portfolio Optimization System
    Main agents package
    \"\"\"
    __version__ = '1.0.0'
"""))

# ── agents/market_data/__init__.py ───────────────────────────
write_file('agents/market_data/__init__.py', textwrap.dedent("""\
    \"\"\"
    Market Data Agent — Week 2
    Foundation agent for data collection and distribution.
    \"\"\"
    from .agent   import MarketDataAgent
    from .storage import MarketDataStorage
    __all__ = ['MarketDataAgent', 'MarketDataStorage']
    __version__ = '1.0.0'
"""))

# ── agents/risk_management/__init__.py ───────────────────────
write_file('agents/risk_management/__init__.py', textwrap.dedent("""\
    \"\"\"
    Risk Management Agent (RiskIQ) — Week 3
    \"\"\"
    from .agent import RiskManagementAgent
    __all__ = ['RiskManagementAgent']
    __version__ = '1.0.0'
"""))

# ── agents/alpha_signal/__init__.py ──────────────────────────
write_file('agents/alpha_signal/__init__.py', textwrap.dedent("""\
    \"\"\"
    Alpha Signal & Portfolio Optimization Agent — Week 4 & 5
    \"\"\"
    from .agent import PortfolioOptimizationAgent
    __all__ = ['PortfolioOptimizationAgent']
    __version__ = '1.0.0'
"""))

print("\n✅ All project files written successfully!")
print("\nFull structure:")
for root, dirs, files in os.walk('agents'):
    dirs[:] = [d for d in dirs if d != '__pycache__']
    level   = root.replace('agents', '').count(os.sep)
    indent  = '  ' * level
    print(f"{indent}📁 {os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  📄 {f}")

---
## 🧪 Section 12: Quick API Test (when API server is running)

In [ ]:
# ── Test the Portfolio Optimization API ─────────────────────
# Run this cell AFTER starting the API server:
#   cd agents/alpha_signal && python api.py

API_BASE = 'http://localhost:8003'

def test_api():
    tests = [
        ('GET',  f'{API_BASE}/',                     None),
        ('POST', f'{API_BASE}/optimize/max-sharpe',  {'symbols': ['AAPL','GOOGL','MSFT'], 'max_weight': 0.4, 'period': '1y'}),
        ('POST', f'{API_BASE}/optimize/min-variance', {'symbols': ['AAPL','GOOGL','MSFT'], 'max_weight': 0.4}),
        ('GET',  f'{API_BASE}/optimize/max-sharpe?symbols=AAPL,MSFT,GOOGL', None),
    ]
    for method, url, payload in tests:
        try:
            r = (requests.post(url, json=payload, timeout=30) if method == 'POST'
                 else requests.get(url, timeout=30))
            d = r.json()
            if r.status_code == 200:
                sharpe = d.get('sharpe_ratio', d.get('status', '?'))
                print(f"  ✅ {method:4s} {url.split('localhost:8003')[1][:40]:40s}  → sharpe/status: {sharpe}")
            else:
                print(f"  ❌ {method:4s} {url[:60]}  → {r.status_code}")
        except requests.exceptions.ConnectionError:
            print(f"  ⚠  API not reachable — start with: cd agents/alpha_signal && python api.py")
            break
        except Exception as e:
            print(f"  ❌ Error: {e}")

test_api()

---
## 📚 Section 13: Mathematical Reference

| Concept | Formula |
|---|---|
| Portfolio Return | $R_p = \mathbf{w}^\top \boldsymbol{\mu} \times 252$ |
| Portfolio Variance | $\sigma_p^2 = \mathbf{w}^\top \boldsymbol{\Sigma}_{annual} \mathbf{w}$ |
| Sharpe Ratio | $S = \frac{R_p - R_f}{\sigma_p}$ |
| VaR (95%) | $\text{VaR}_{95} = -\text{percentile}(r, 5)$ |
| CVaR (95%) | $\text{CVaR}_{95} = E[r \mid r \le -\text{VaR}_{95}]$ |
| Max Drawdown | $\text{MDD} = \min_t \frac{P_t - \max_{s\le t} P_s}{\max_{s\le t} P_s}$ |
| RSI | $RSI = 100 - \frac{100}{1 + \bar{G}/\bar{L}}$ |
| Bollinger %B | $\%B = \frac{P - \text{Lower}}{\text{Upper} - \text{Lower}}$ |

**Optimization Problem:**

$$\max_{\mathbf{w}} \frac{\mathbf{w}^\top \boldsymbol{\mu} - R_f}{\sqrt{\mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w}}} \quad \text{s.t.} \quad \mathbf{1}^\top \mathbf{w} = 1, \quad w_{\min} \le w_i \le w_{\max}$$

Solved using **SciPy SLSQP** (Sequential Least Squares Programming).

---

## 🚀 Section 14: Quick Start Guide

```bash
# 1. Install dependencies
pip install -r requirements.txt

# 2. Start Market Data API (Terminal 1)
cd agents/market_data && python api.py       # port 8000

# 3. Start Risk Management API (Terminal 2)
cd agents/risk_management && python api.py   # port 8001

# 4. Start Portfolio Optimization API (Terminal 3)
cd agents/alpha_signal && python api.py      # port 8003

# 5. Run this notebook or launch the Gradio dashboard
jupyter notebook Portfolio_Optimization_Agent_Week5.ipynb
```

---
*⬡ Portfolio Intelligence System · IIT Madras 2026 · Week 5 of 16*  
*Team: Ashwini · Dibyendu Sarkar · Jyoti Ranjan Sethi*  
*⚠️ For educational purposes only — not financial advice.*